# TexTag: Transfer Learning with ResNet18

Instead of learning filters from scratch (baseline: 28.4% val accuracy), start from a ResNet18
that already learned visual features from 1.2 million ImageNet images, and adapt it to our 47 textures.

**Run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

Sections:
1. Setup
2. Data
3. Model
4. Phase 1: feature extractor (train only the last layer)
5. Phase 2: fine-tuning (train all layers)
6. Comparison
7. Save the final model

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/textag"):
        !git clone -q https://github.com/codecustard/textag.git /content/textag
    %cd /content/textag/notebooks

In [ ]:
import copy
from pathlib import Path
from timeit import default_timer as timer

import torch
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from textag.data import loaders, class_names
from textag.model import resnet18, unfreeze, n_params
from textag import engine

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = os.environ.get("TEXTAG_EPOCHS")   # env var only used for quick test runs

In [ ]:
def plot_curves(hist, title):
    """Train vs val loss and accuracy per epoch; dashed line = best epoch."""
    h = pd.DataFrame({k: v for k, v in hist.items() if k != "best_epoch"})
    h.index = h.index + 1
    best = hist["best_epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["loss", "acc"]):
        ax.plot(h.index, h[f"train_{metric}"], label="train")
        ax.plot(h.index, h[f"val_{metric}"], label="val")
        ax.axvline(best, color="gray", linestyle="--", label=f"best epoch ({best})")
        ax.set_xlabel("epoch")
        ax.set_title({"loss": "Loss (lower is better)", "acc": "Accuracy (higher is better)"}[metric])
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 2. Data

Same data pipeline as the baseline. Images are already normalized with the ImageNet mean/std,
which is what the pretrained ResNet expects (checked in the EDA).

In [ ]:
dl = loaders(batch_size=32, num_workers=2)
classes = class_names()
print({s: len(d.dataset) for s, d in dl.items()})

## 3. Model

**Transfer learning:** reuse a model trained on a big dataset as the starting point for a new task.
ResNet18's early layers already detect edges, colors and textures, which is exactly what we need.
We only replace its **last layer**: 512 features → 1000 ImageNet classes becomes 512 features → our 47 classes.

**Why ResNet18**

| Reason | Detail |
|---|---|
| Little data | 40 images per class is too few to learn filters from scratch (baseline: 28.4%); pretrained filters come from 1.2M images |
| Small and fast | ~11M weights: quick to train, fast single-image predictions on a CPU (matters for the web app) |
| Standard | Built into `torchvision` with pretrained weights; widely used and easy to cite |

**Two phases**

| | Phase 1: feature extractor | Phase 2: fine-tuning |
|---|---|---|
| What trains | Only the new last layer (the rest is frozen) | All layers |
| Idea | Use ImageNet's features as they are | Gently adjust all filters toward textures |
| Learning rate | 1e-3 | 1e-4 (10× smaller, so we don't wreck what it already knows) |

**References**

- He et al. (2015). *Deep Residual Learning for Image Recognition* (ResNet). [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)
- Yosinski et al. (2014). *How transferable are features in deep neural networks?* [arXiv:1411.1792](https://arxiv.org/abs/1411.1792)

In [ ]:
model = resnet18(num_classes=len(classes), pretrained=True, freeze_backbone=True)
print("new last layer:", model.fc)
print(f"trainable weights (frozen backbone): {n_params(model):,}")
print(f"total weights:                       {sum(p.numel() for p in model.parameters()):,}")

## 4. Phase 1: feature extractor

Only the new last layer trains. Same choices as the baseline: `CrossEntropyLoss`, `AdamW`,
`ReduceLROnPlateau` (halve the LR if val loss stalls for 3 epochs), early stopping (patience 5).
Up to 15 epochs. The optimizer only gets the trainable weights.

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

start_time = timer()
hist1 = engine.train(model=model,
                     train_dataloader=dl["train"],
                     val_dataloader=dl["val"],
                     optimizer=optimizer,
                     loss_fn=loss_fn,
                     epochs=int(EPOCHS or 15),
                     device=device,
                     scheduler=scheduler,
                     patience=5)
time1 = timer() - start_time
print(f"Phase 1 training time: {time1:.1f} seconds")

phase1_weights = copy.deepcopy(model.state_dict())   # keep, in case phase 2 doesn't help

In [ ]:
plot_curves(hist1, "Phase 1: feature extractor")

## 5. Phase 2: fine-tuning

Unfreeze every layer and keep training from the best phase-1 weights, with a 10× smaller learning rate.
Up to 20 epochs, early stopping (patience 5).

In [ ]:
unfreeze(model)
print(f"trainable weights (all layers): {n_params(model):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

start_time = timer()
hist2 = engine.train(model=model,
                     train_dataloader=dl["train"],
                     val_dataloader=dl["val"],
                     optimizer=optimizer,
                     loss_fn=loss_fn,
                     epochs=int(EPOCHS or 20),
                     device=device,
                     scheduler=scheduler,
                     patience=5)
time2 = timer() - start_time
print(f"Phase 2 training time: {time2:.1f} seconds")

In [ ]:
plot_curves(hist2, "Phase 2: fine-tuning")

## 6. Comparison

Validation accuracy at each model's best epoch. (The test set stays untouched until the evaluation step.)

In [ ]:
BASELINE_VAL_ACC = 0.284   # SmallCNN from scratch, from 02_baseline.ipynb

def best(hist):
    i = hist["best_epoch"] - 1
    return hist["train_acc"][i], hist["val_acc"][i]

p1_train, p1_val = best(hist1)
p2_train, p2_val = best(hist2)

comparison = pd.DataFrame({
    "model": ["SmallCNN (from scratch)", "ResNet18 phase 1 (feature extractor)", "ResNet18 phase 2 (fine-tuned)"],
    "train acc": [None, p1_train, p2_train],
    "val acc": [BASELINE_VAL_ACC, p1_val, p2_val],
    "training time (s)": [1096, round(time1), round(time2)],
})
comparison.style.format({"train acc": "{:.1%}", "val acc": "{:.1%}"}, na_rep="-")

In [ ]:
plt.figure(figsize=(7, 3.5))
bars = plt.bar(["SmallCNN\n(scratch)", "ResNet18\nphase 1", "ResNet18\nphase 2"], comparison["val acc"])
plt.bar_label(bars, labels=[f"{v:.1%}" for v in comparison["val acc"]])
plt.axhline(1 / len(classes), color="gray", linestyle="--", label="random guessing")
plt.ylabel("val accuracy")
plt.ylim(0, 1)
plt.title("Validation accuracy")
plt.legend()
plt.show()

## 7. Save the final model

Keep whichever phase scored higher on validation accuracy. This file is what the web app loads.

In [ ]:
if p1_val > p2_val:
    model.load_state_dict(phase1_weights)
    chosen = "phase 1"
else:
    chosen = "phase 2"

MODEL_PATH = ROOT / "models" / "textag.pt"
MODEL_PATH.parent.mkdir(exist_ok=True)
torch.save({"arch": "resnet18", "state_dict": model.state_dict(), "classes": classes}, MODEL_PATH)
print(f"saved {MODEL_PATH.name} ({chosen}), {MODEL_PATH.stat().st_size / 1e6:.0f} MB")

if IN_COLAB:   # download the weights so they can go into the repo's models/ folder
    from google.colab import files
    files.download(str(MODEL_PATH))

**Findings:** _filled in after the full run._

**Next:** evaluate the final model on the test set (accuracy, precision/recall/F1, confusion matrix).